# Module 1: From Incident Records to a Time Series

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Public safety data does not arrive as a time series. It arrives as a pile of
individual records, one row per incident, in no particular order. Turning that
pile into a series is the first thing any analysis does, and every choice made
along the way changes what the series can show.

In this notebook you will take 22,233 individual use of force records and build
the exact twelve numbers that Beginner Topic 1 opened with. Then you will build
two completely different series from the same records, to make the point that
there is no such thing as *the* time series for a dataset.

**About 15 minutes. No prior pandas experience assumed.**

## 1. Load the records

One row per use of force incident, across twelve agencies and seven and a half
years.

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
from pathlib import Path

import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

print("reading from:", BASE)

uof = pd.read_csv(BASE + "use_of_force.csv")
print(f"{len(uof):,} records, {uof.shape[1]} columns")
uof.head(3)

Look at what one row is: an incident, at an agency, on a date, with details
about the subject, the officer, the resistance and the force used.

Nothing here is a time series yet. There is no ordering and no time unit.

In [ ]:
uof[["incident_id", "agency_id", "incident_date", "incident_type",
     "subject_resistance", "type_of_force"]].head(5)

## 2. Narrow to one agency and one year

A series is always a series *of something*. Decide what that something is before
grouping anything.

In [ ]:
g = uof[(uof["agency_id"] == "A012") & (uof["year_month"].str[:4] == "2023")]
print(f"Grandview Police Department, 2023: {len(g):,} records")

## 3. Choose the time key

The file gives two options. `incident_date` is the exact day. `year_month` is
the month it falls in.

The `YYYY-MM` format is worth a moment. Because the month is zero padded, these
strings sort in the correct order as plain text, so grouping on them gives
months in sequence with no extra work. A format like `1/7/2023` would not.

In [ ]:
print(sorted(g["year_month"].unique()))

For anything beyond sorting, convert to a proper period. This is what lets you
reindex to a complete calendar later, which is Module 2.

In [ ]:
g = g.copy()
g["period"] = pd.PeriodIndex(g["year_month"], freq="M")
print(g["period"].min(), "to", g["period"].max())

## 4. Group

`size()` counts the rows in each group. That is the whole operation: 1,136
records become 12 numbers.

In [ ]:
monthly = g.groupby("year_month").size()
print(monthly.to_string())

Those are the numbers Beginner Topic 1 opened with. They were never handed down
from anywhere; they are the result of two decisions, which agency and which time
unit, applied to a file of individual records.

## 5. Check the work against the published table

The repository also ships `agency_monthly.csv`, which carries the same counts.
Whenever a prepared table exists, rebuild it once from the raw records and
confirm the two agree. If they do not, one of them is wrong and it is better to
find out now.

In [ ]:
am = pd.read_csv(BASE + "agency_monthly.csv")
published = (am[(am["agency_id"] == "A012") & (am["year_month"].str[:4] == "2023")]
             .set_index("year_month")["n_uof"])

check = pd.DataFrame({"rebuilt": monthly, "published": published})
check["match"] = check["rebuilt"] == check["published"]
print(check.to_string())
print("\nall twelve months agree:", check["match"].all())

## 6. The same records, grouped differently

Here is the part that matters. Nothing about those 1,136 records forced the
series above. Group them another way and you get a different series, equally
valid, answering a different question.

**By incident type:**

In [ ]:
by_type = (g.groupby(["year_month", "incident_type"]).size()
             .unstack(fill_value=0))
by_type[["Offense Against Person", "Property Offense", "Vehicle Stop"]]

**By outcome:**

In [ ]:
by_injury = (g.groupby(["year_month", "subject_injury"]).size()
               .unstack(fill_value=0))
by_injury["share injured"] = (100 * by_injury["Yes"] /
                              by_injury.sum(axis=1)).round(1)
by_injury

Three series, one file. The monthly total rose into July. The share of incidents
involving injury did something else entirely. Neither is more correct; they
answer different questions, and the question has to come first.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
monthly.plot(ax=ax[0], marker="o", color="#2a78d6")
ax[0].set_title("total incidents", loc="left")
ax[0].set_ylim(0, None)
by_injury["share injured"].plot(ax=ax[1], marker="o", color="#eb6834")
ax[1].set_title("share of incidents with a subject injury, percent", loc="left")
ax[1].set_ylim(0, None)
for a in ax:
    a.tick_params(axis="x", rotation=60, labelsize=8)
    a.set_xlabel("")
plt.tight_layout()

## 7. Counting events is not counting people

A subject can appear in more than one incident. If the question is about people
rather than events, de duplicate first. The two answers are different, and
neither is wrong until the question is stated.

In [ ]:
print(f"incident records : {len(uof):,}")
print(f"distinct subjects: {uof['subject_id'].nunique():,}")

repeats = uof["subject_id"].value_counts()
print(f"subjects appearing more than once: {(repeats > 1).sum():,}")
print(f"most incidents for one subject   : {repeats.max()}")

## 8. What to carry away

| Decision | Made when | Changes |
|---|---|---|
| Which agency, or all of them | before grouping | who the series describes |
| Which time unit | before grouping | what patterns can appear at all |
| Which grouping columns | at the group step | what question the series answers |
| Events or people | at the count step | the size of every number |

A series is a set of choices applied to records. Write the choices down next to
the chart, because a reader cannot recover them from the numbers.

## Exercise

Build a monthly series of **use of force incidents involving a firearm
discharge, across all twelve agencies, for 2024 and 2025**, and say which month
was highest.

Fill in the blanks below.

In [ ]:
# Fill in the two blanks, then run this cell.
FORCE_TYPE = None          # try "Firearm Discharge"
YEARS = None               # try ["2024", "2025"]

if FORCE_TYPE and YEARS:
    subset = uof[(uof["type_of_force"] == FORCE_TYPE) &
                 (uof["year_month"].str[:4].isin(YEARS))]
    series = subset.groupby("year_month").size()
    print(series.to_string())
    print("highest month:", series.idxmax(), series.max())
else:
    print("Set FORCE_TYPE and YEARS above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
subset = uof[(uof["type_of_force"] == "Firearm Discharge") &
             (uof["year_month"].str[:4].isin(["2024", "2025"]))]

series = subset.groupby("year_month").size()
print(series.to_string())
print("highest month:", series.idxmax(), series.max())
```

Two things to notice in the answer. First, the counts are small, often single
digits, so the highest month says very little on its own. That is the subject of
Module 4. Second, some months may be absent from the result entirely, because a
group by never produces a row for a month in which nothing happened. That is the
subject of Module 2, next.

</details>

---

**Next:** [Module 2, Building an Honest Calendar](Module_02_Building_An_Honest_Calendar.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*